# Stage B v6 — four-pass Swiss law citation expert

**Paradigm shift**: binary keep/reject -> continuous scoring + adaptive K per query.

**Pipeline**:
1. Pass-1 **Issue Map** — for each legal issue, list must_cite / should_cite / procedural / precedents
2. Pass-2 **Continuous Scoring** — per candidate, produce must_cite_score / should_cite_score / supporting_relevance
3. Pass-3 **K Predictor** — given top-50 scored candidates per query, predict how many a thorough opinion would cite
4. Pass-4 **Diversified Selection** — local; pick top-K_predicted with per-issue floor + score floor 0.40

**Stage A ceiling at TOP_K=2000**: macro F1 = 0.776 (oracle picking)
**v5.3 baseline**: macro K-cap F1 = 0.107
**v6 target**: macro F1 0.45-0.55 (single-step jump). Path to 0.65+ then needs reranker fine-tune.

## Phase 0 — Setup

In [ ]:
import os, sys, subprocess, json, time, gc, io, re, math
from pathlib import Path
sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding="utf-8", errors="replace")

IS_COLAB = "google.colab" in sys.modules
print(f"Colab: {IS_COLAB}")
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    subprocess.run(["pip","install","-q","-U",
        "vllm>=0.9.1","transformers>=4.51.0","pandas==2.2.3",
        "pyarrow==16.1.0","numpy==1.26.4","tqdm"], check=True)


## Phase 1 — Paths and config

In [ ]:
DRIVE_ROOT  = Path("/content/drive/MyDrive/swiss_law")
STAGE_B_IN  = DRIVE_ROOT / "research" / "stage_b_input" / "stage_b_input.parquet"
VAL_CSV     = DRIVE_ROOT / "data"     / "val.csv"
GOLD_SETS   = DRIVE_ROOT / "research" / "anchor_funnel_val001_v7" / "snapshot" / "gold_doc_sets.json"
VAL_ASPECTS = DRIVE_ROOT / "research" / "concept_embedding_path" / "multi_aspect" / "val_aspects.parquet"

OUT_DIR = DRIVE_ROOT / "research" / "stage_b_grounded_llm" / "v6_all_queries"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TOP_K              = 2000
LLM_MODEL          = "Qwen/Qwen3-32B-AWQ"
PASS1_SEEDS        = [42, 43, 44]
PASS1_MAX_TOKENS   = 4096
PASS2_MAX_TOKENS   = 1024
PASS3_MAX_TOKENS   = 2048
MAX_MODEL_LEN      = 8192
SCORE_FLOOR        = 0.40  # candidates with composite < 0.40 are dropped even if rank < K

print(f"Output dir: {OUT_DIR}")
print("\nVerifying inputs:")
for p in [STAGE_B_IN, VAL_CSV, GOLD_SETS, VAL_ASPECTS]:
    print(f"  {'OK' if p.exists() else 'MISSING'}  {p}")


## Phase 2 — Load data + tier candidates

In [ ]:
import pandas as pd

val_df = pd.read_csv(VAL_CSV)
QIDS = sorted(val_df.query_id.unique().tolist())
qid_to_query = {r.query_id: str(r.query) for r in val_df.itertuples()}

asp_df = pd.read_parquet(VAL_ASPECTS)
qid_to_aspects = {r.query_id: list(r.aspects) for r in asp_df.itertuples()}

gold_sets = json.load(open(GOLD_SETS, encoding="utf-8"))
qid_to_gold = {q: set(gold_sets.get(q, [])) for q in QIDS}
gold_totals = {q: len(qid_to_gold[q]) for q in QIDS}

sb_all = pd.read_parquet(STAGE_B_IN)
sb_per_query = {}
for q in QIDS:
    sub = (sb_all[sb_all.qid == q].sort_values("stage_a_rank").head(TOP_K).reset_index(drop=True))
    sub["tier"] = "drop"
    _auto = sub.article_match & ((sub.co_citation_count >= 30) | sub.code_in_target)
    sub.loc[_auto, "tier"] = "auto"
    _mid = (~_auto) & (sub.article_match | (sub.co_citation_count >= 5) | (sub.concept_cosine_score >= 0.55))
    sub.loc[_mid, "tier"] = "llm"
    sb_per_query[q] = sub

print(f"{'qid':<8} {'gold':>4} {'in_topk':>8} {'auto':>5} {'llm':>5} {'drop':>5}")
for q in QIDS:
    s = sb_per_query[q]
    print(f"  {q:<6} {gold_totals[q]:>4} {int(s.is_gold.sum()):>8} "
          f"{int((s.tier=='auto').sum()):>5} {int((s.tier=='llm').sum()):>5} {int((s.tier=='drop').sum()):>5}")


## Phase 3 — Pass-1 Issue Map

For each query: ask Qwen3 to factor the question into legal issues and produce a structured authority map per issue. 3 samples per query for coverage.

In [ ]:
PASS1_PROMPT = """You are a senior Swiss attorney with mastery of Swiss federal law (ZGB, OR, StGB, StPO, ZPO, BGG, SchKG, BV, EMRK) and the BGE jurisprudence. You are about to write a legal opinion on the question below. Before drafting, you map the legal landscape: identify the issues the opinion must address, and list the canonical Swiss authorities for each.

CRITICAL CITATION RULES:
- Cite ONLY Swiss legal provisions you are confident actually exist. Never invent article numbers.
- Code-size limits:
    ZGB:   articles 1-977
    OR:    articles 1-1186
    StGB:  articles 1-393
    StPO:  articles 1-457
    ZPO:   articles 1-408
    BGG:   articles 1-132
    SchKG: articles 1-340
    BV:    articles 1-197
    EMRK:  articles 1-59
- If you do not know the exact article number, OMIT it. Sparse and accurate beats long and fabricated.

LEGAL QUESTION FROM CLIENT:
{question}

ANALYST'S ASPECT DECOMPOSITION (informational — you may restructure as you see fit):
{aspects_block}

YOUR TASK:
Produce a structured issue map. For each legal ISSUE the opinion must address:
1. State the controlling rule in one sentence (your own words, not a citation).
2. List MUST_CITE authorities — provisions or BGE references where OMISSION would be malpractice. These are the foundational rules + leading precedents directly on point. Be selective: 3-8 entries.
3. List SHOULD_CITE authorities — a thorough opinion includes these as supporting/related; omission is suboptimal but not malpractice. 3-10 entries.
4. List PROCEDURAL_RULES — articles routinely cited for the procedural posture of this question (e.g., Art. 100 BGG for any Federal Tribunal appeal, cost rules, brief requirements).
5. List LEADING_PRECEDENTS — specific BGE references that established the doctrine. Omit if uncertain of the BGE number.

OUTPUT STRICT JSON (no prose, begin with `{{`):
{{
  "issues": [
    {{
      "id": "i1",
      "label": "<short label>",
      "rule_statement": "<one sentence stating the controlling rule>",
      "must_cite_authorities": [{{"cit": "Art. 285 ZGB", "why": "establishes child support measurement standard"}}],
      "should_cite_authorities": [{{"cit": "Art. 286 ZGB", "why": "modification and indexing"}}],
      "procedural_rules": [{{"cit": "Art. 100 BGG", "why": "Federal Tribunal appeal deadline"}}],
      "leading_precedents": [{{"cit": "BGE 137 III 118", "doctrine": "hypothetical income may be imputed to non-earning parents"}}]
    }},
    ...
  ]
}}
"""

def format_aspects_block(aspects):
    parts = []
    for a in aspects:
        aid = a.get("id","")
        lbl = a.get("label","")
        w   = float(a.get("weight",0))
        terms = list(a.get("concepts_en",[])) + list(a.get("terms_de",[]))[:4]
        parts.append(f"  {aid} (weight={w:.2f}): {lbl}    [terms: {', '.join(terms)}]")
    return "\n".join(parts)

pass1_prompts, pass1_meta = [], []
for q in QIDS:
    prompt = PASS1_PROMPT.format(
        question=qid_to_query[q],
        aspects_block=format_aspects_block(qid_to_aspects.get(q, [])),
    )
    for seed in PASS1_SEEDS:
        pass1_prompts.append(prompt)
        pass1_meta.append((q, seed))
print(f"Pass-1 prompts: {len(pass1_prompts)}")
print(f"Sample (first 1200 chars):\n{pass1_prompts[0][:1200]}")


## Phase 4 — Run Pass-1

In [ ]:
from vllm import LLM, SamplingParams

print(f"Loading {LLM_MODEL} ...")
llm = LLM(model=LLM_MODEL, dtype="bfloat16",
          gpu_memory_utilization=0.60, max_model_len=MAX_MODEL_LEN,
          enforce_eager=False)

sp_list = [SamplingParams(temperature=0.6, top_p=0.95, top_k=20, min_p=0.0,
                          max_tokens=PASS1_MAX_TOKENS, seed=seed)
           for _, seed in pass1_meta]

print(f"Pass-1: {len(pass1_prompts)} prompts ...")
t0 = time.time()
outs1 = llm.chat([[{"role":"user","content":p}] for p in pass1_prompts], sampling_params=sp_list)
pass1_raws = [o.outputs[0].text for o in outs1]
print(f"Done in {(time.time()-t0)/60:.1f} min")


## Phase 5 — Parse + union issue maps per query

In [ ]:
def parse_json_block(raw):
    s = raw.strip()
    s = re.sub(r"<think>.*?</think>", "", s, flags=re.DOTALL).strip()
    if s.startswith("```"):
        s = s.split("\n", 1)[-1]
        if s.endswith("```"):
            s = s.rsplit("```", 1)[0]
    a, b = s.find("{"), s.rfind("}")
    if a == -1 or b == -1: return None
    for cand in [s[a:b+1], s[a:b+1].replace(",}", "}").replace(",]", "]")]:
        try: return json.loads(cand)
        except Exception: pass
    return None

def coerce_item(it):
    if isinstance(it, str):
        return {"cit": it, "why": ""}
    if isinstance(it, dict):
        cit = it.get("cit") or it.get("citation") or it.get("article") or ""
        why = it.get("why") or it.get("doctrine") or it.get("role") or it.get("topic") or ""
        return {"cit": str(cit), "why": str(why)}
    return None

def norm_cit(c):
    s = re.sub(r"\bAbs\.\s*\d+\w*\b", "", c or "", flags=re.IGNORECASE)
    s = re.sub(r"\blit\.\s*\w+\b", "", s, flags=re.IGNORECASE)
    s = re.sub(r"[.,]", "", s)
    return " ".join(s.lower().split())

ART_LIMITS = {"ZGB":977,"OR":1186,"StGB":393,"StPO":457,"ZPO":408,
              "BGG":132,"SchKG":340,"BV":197,"EMRK":59}
ART_RE = re.compile(r"Art\.\s*(\d+)\s*(?:Abs\.\s*\d+\w*\s*)?(?:lit\.\s*\w+\s*)?(\w+)")

def is_valid(cit):
    m = ART_RE.search(cit or "")
    if not m: return True
    num, code = int(m.group(1)), m.group(2)
    return num <= ART_LIMITS.get(code, 99999)

CATS = ("must_cite_authorities","should_cite_authorities","procedural_rules","leading_precedents")

qid_to_issue_maps = {q: [] for q in QIDS}
parse_fail = 0
for (qid, seed), raw in zip(pass1_meta, pass1_raws):
    parsed = parse_json_block(raw)
    if parsed is None or "issues" not in parsed:
        parse_fail += 1
        continue
    qid_to_issue_maps[qid].append(parsed)
print(f"Parsed {sum(len(v) for v in qid_to_issue_maps.values())} / {len(pass1_meta)} issue maps; {parse_fail} fail")

# Merge per query: align issues across samples by label similarity (best-effort),
# union authority lists per category. Conservative: keep all issues from each sample.
qid_to_issues = {}
total_stripped = 0
total_coerced = 0
for q in QIDS:
    samples = qid_to_issue_maps[q]
    if not samples:
        qid_to_issues[q] = []
        continue
    # Use first sample as the canonical issue structure (model usually produces 3-5 issues)
    canonical = samples[0].get("issues", [])
    # For each canonical issue, find best-match issues in other samples by label substring
    merged = []
    for i, ci in enumerate(canonical):
        iid = ci.get("id", f"i{i+1}")
        ilabel = (ci.get("label","") or "").lower()
        pooled = {c: {} for c in CATS}
        # Seed pooled with canonical
        for cat in CATS:
            for raw_it in (ci.get(cat, []) or []):
                if isinstance(raw_it, str): total_coerced += 1
                it = coerce_item(raw_it)
                if it is None or not it["cit"]: continue
                k = norm_cit(it["cit"])
                if k not in pooled[cat]:
                    pooled[cat][k] = it
        # Add other samples' contributions on label match
        for s_idx, other in enumerate(samples[1:]):
            for oi in other.get("issues", []):
                olabel = (oi.get("label","") or "").lower()
                # match if labels share a substantive word
                if not ilabel or not olabel: continue
                shared = set(ilabel.split()) & set(olabel.split())
                shared = {w for w in shared if len(w) > 4}
                if not shared: continue
                for cat in CATS:
                    for raw_it in (oi.get(cat, []) or []):
                        if isinstance(raw_it, str): total_coerced += 1
                        it = coerce_item(raw_it)
                        if it is None or not it["cit"]: continue
                        k = norm_cit(it["cit"])
                        if k not in pooled[cat]:
                            pooled[cat][k] = it
        # Strip hallucinated articles
        final = {"id": iid, "label": ci.get("label",""), "rule_statement": ci.get("rule_statement","")}
        for cat in CATS:
            items = list(pooled[cat].values())
            kept = [it for it in items if is_valid(it.get("cit",""))]
            total_stripped += len(items) - len(kept)
            final[cat] = kept
        merged.append(final)
    qid_to_issues[q] = merged

print(f"Coerced {total_coerced} plain-string items; stripped {total_stripped} hallucinated refs")
print(f"\nIssue counts per query:")
for q in QIDS:
    issues = qid_to_issues[q]
    n_auth = sum(len(iss.get(c,[])) for iss in issues for c in CATS)
    print(f"  {q}: {len(issues)} issues, {n_auth} total authorities  "
          f"(must={sum(len(iss.get('must_cite_authorities',[])) for iss in issues)}, "
          f"should={sum(len(iss.get('should_cite_authorities',[])) for iss in issues)})")


## Phase 6 — Build Pass-2 prompts (continuous scoring)

In [ ]:
def render_issue_map(issues):
    if not issues: return "(no issue map — Pass-1 failed for this query)"
    chunks = []
    for iss in issues:
        chunks.append(f"== Issue {iss.get('id','?')}: {iss.get('label','')} ==")
        rs = iss.get("rule_statement","")
        if rs: chunks.append(f"  Rule: {rs}")
        for cat, hdr in [("must_cite_authorities","MUST-CITE (omission = malpractice)"),
                         ("should_cite_authorities","Should-cite (thorough opinion includes)"),
                         ("procedural_rules","Procedural rules"),
                         ("leading_precedents","Leading BGE precedents")]:
            items = iss.get(cat, [])
            if not items: continue
            chunks.append(f"  {hdr}:")
            for it in items:
                chunks.append(f"    - {it.get('cit',''):<25}  {it.get('why','')[:100]}")
    return "\n".join(chunks)

PASS2_PROMPT = """You are a senior Swiss lawyer evaluating one candidate citation for inclusion in your legal opinion. You have already mapped the issues and canonical authorities (the ISSUE MAP below). Now score this candidate on three continuous dimensions.

LEGAL QUESTION:
{question}

ISSUE MAP (your prior research):
{issue_map}

CANDIDATE:
- Citation: {citation}
- Type: {family_label}  ({family_extra})
- Paragraph role: {role}
- Substantive text (original language):
---BEGIN---
{text}
---END---

RETRIEVAL EVIDENCE (advisory):
- Query names this article: {article_match}  | co-citations: {co_citation_count}  | concept-cosine: {concept_cosine_score:.2f}  | code matches area: {code_in_target}

SCORING DIMENSIONS (each independently 0.0-1.0):

A. must_cite_score — Would OMITTING this citation be a serious defect in the opinion?
   1.0 = yes, this is foundational/controlling for one of the issues (e.g., the candidate's citation literally appears in MUST-CITE list, OR its text states the controlling rule for one of the issues)
   0.7 = likely yes, strong overlap with foundational doctrine
   0.4 = arguable
   0.0 = no, opinion is fine without it

B. should_cite_score — Would a THOROUGH opinion include this citation (even if omission isn't malpractice)?
   1.0 = yes, listed in SHOULD-CITE or text supports an issue's argument substantially
   0.5 = peripheral
   0.0 = no

C. supporting_relevance — Does the text content directly serve any legal argument the opinion will make?
   1.0 = directly states a rule or holding that the opinion will rely on
   0.5 = mentions the topic but doesn't state the rule
   0.0 = unrelated text

Also classify the citation_role: foundational | procedural | supporting | tangential | off_topic
And identify the matched issue id (or "none").

BE CALIBRATED: only ~10-20% of candidates should have any score >= 0.7. Most should be <= 0.4.

OUTPUT STRICT JSON (no prose):
{{"must_cite_score": <0.0-1.0>, "should_cite_score": <0.0-1.0>, "supporting_relevance": <0.0-1.0>, "matched_issue": "<i1|i2|...|none>", "citation_role": "<foundational|procedural|supporting|tangential|off_topic>", "reasoning": "<one sentence>"}}
"""

def family_label(r): return "Swiss court precedent paragraph" if r.family=="court" else "Swiss statutory provision"
def family_extra(r):
    if r.family=="court": return f"Court base: {r.court_base!s}, Chamber: {r.chamber!s}"
    return f"Code: {r.law_code!s}, Law: {r.law_title!s}"

def build_pass2(r, q, issue_map_text):
    return PASS2_PROMPT.format(
        question=qid_to_query[q][:1500],
        issue_map=issue_map_text,
        citation=r.citation,
        family_label=family_label(r),
        family_extra=family_extra(r),
        role=r.role or "(unknown)",
        article_match=str(bool(r.article_match)).lower(),
        co_citation_count=int(r.co_citation_count),
        concept_cosine_score=float(r.concept_cosine_score),
        code_in_target=str(bool(r.code_in_target)).lower(),
        text=(r.text or "")[:1200].replace('"', "'"),
    )

pass2_prompts, pass2_meta = [], []
for q in QIDS:
    issue_map_text = render_issue_map(qid_to_issues.get(q, []))
    sb_llm = sb_per_query[q][sb_per_query[q].tier == "llm"].reset_index(drop=True)
    for r in sb_llm.itertuples():
        pass2_prompts.append(build_pass2(r, q, issue_map_text))
        pass2_meta.append((q, r.did))

print(f"Pass-2 prompts: {len(pass2_prompts)}")
print(f"Mean prompt length: {int(sum(len(p) for p in pass2_prompts)/max(1,len(pass2_prompts)))}")


## Phase 7 — Run Pass-2

In [ ]:
sp_pass2 = SamplingParams(temperature=0.6, top_p=0.95, top_k=20, min_p=0.0,
                          max_tokens=PASS2_MAX_TOKENS, seed=42)

print(f"Pass-2: {len(pass2_prompts):,} prompts ...")
t0 = time.time()
outs2 = llm.chat([[{"role":"user","content":p}] for p in pass2_prompts], sampling_params=sp_pass2)
pass2_raws = [o.outputs[0].text for o in outs2]
print(f"Done in {(time.time()-t0)/60:.1f} min")


## Phase 8 — Parse continuous scores + compute composite

In [ ]:
def parse_scores(raw):
    parsed = parse_json_block(raw) or {}
    def f(k, default=0.0):
        try: return float(parsed.get(k, default) or default)
        except Exception: return default
    must = max(0.0, min(1.0, f("must_cite_score")))
    should = max(0.0, min(1.0, f("should_cite_score")))
    support = max(0.0, min(1.0, f("supporting_relevance")))
    return {
        "must_cite_score": must,
        "should_cite_score": should,
        "supporting_relevance": support,
        "composite": max(must, should, 0.5 * support),
        "matched_issue": str(parsed.get("matched_issue","") or "")[:8],
        "citation_role": str(parsed.get("citation_role","") or "")[:30],
        "reasoning": str(parsed.get("reasoning","") or "")[:300],
        "parse_ok": bool(parsed),
    }

def auto_composite(cc):
    # Bounded above by foundational scores: 0.5-0.7 range
    return min(0.70, 0.50 + 0.20 * math.log(max(1, cc) + 1) / 10.0)

pass2_lookup = {(q, d): r for (q, d), r in zip(pass2_meta, pass2_raws)}

all_rows = []
for q in QIDS:
    sb = sb_per_query[q]
    # AUTO tier: composite from co_citation_count, in [0.5, 0.7]
    for r in sb[sb.tier=="auto"].itertuples():
        comp = auto_composite(int(r.co_citation_count))
        all_rows.append({
            "qid": q, "did": r.did, "is_gold": bool(r.is_gold), "tier": "auto",
            "citation": r.citation, "family": r.family, "stage_a_rank": int(r.stage_a_rank),
            "must_cite_score": 0.0, "should_cite_score": 0.0, "supporting_relevance": 0.0,
            "composite": comp,
            "matched_issue": "auto", "citation_role": "auto_dossier",
            "reasoning": f"dossier (cc={int(r.co_citation_count)})",
            "parse_ok": True,
        })
    # LLM tier
    for r in sb[sb.tier=="llm"].itertuples():
        raw = pass2_lookup.get((q, r.did), "")
        scores = parse_scores(raw)
        all_rows.append({
            "qid": q, "did": r.did, "is_gold": bool(r.is_gold), "tier": "llm",
            "citation": r.citation, "family": r.family, "stage_a_rank": int(r.stage_a_rank),
            **scores,
        })

out_df = pd.DataFrame(all_rows)
print(f"Total rows: {len(out_df)}")
print(f"Per-tier composite distribution:")
print(out_df.groupby("tier")["composite"].describe()[["mean","50%","75%","max"]])

# Diagnostic: are scores well-spread or bunched?
print(f"\\nLLM-tier composite histogram (10 bins):")
import numpy as np
llm_comp = out_df[out_df.tier=="llm"].composite
print(pd.cut(llm_comp, bins=np.linspace(0,1,11)).value_counts().sort_index())

# Diagnostic: gold vs non-gold composite separation
gold = out_df[out_df.is_gold]
non = out_df[~out_df.is_gold]
print(f"\\nGold mean composite:    {gold.composite.mean():.3f}  (n={len(gold)})")
print(f"Non-gold mean composite: {non.composite.mean():.3f}  (n={len(non)})")
print(f"Separation:               {gold.composite.mean()-non.composite.mean():+.3f}")


## Phase 9 — Build Pass-3 prompts (K predictor per query)

In [ ]:
PASS3_PROMPT = """You are a senior Swiss attorney. You have completed your research and now need to decide how many citations to include in the legal opinion.

LEGAL QUESTION:
{question}

ISSUE MAP:
{issue_map}

TOP-50 CANDIDATES BY RELEVANCE SCORE (these are the most relevant sources we've identified, score is your prior judgment on a 0-1 scale where 1.0 = must-cite):
{candidates_block}

YOUR TASK:
Decide how many citations a THOROUGH but not excessive Swiss legal opinion on this question should include. Consider:
- Number of distinct legal issues (each issue needs its foundational + procedural citations)
- Question complexity (multi-issue, novel, contested -> more citations; simple application -> fewer)
- Swiss legal-writing norms: a tight opinion on a narrow question may cite 5-10 sources; a complex multi-issue opinion may cite 30-50 sources

Output K (total citations) and minimum_per_issue (smallest number to allocate to any one issue).

OUTPUT STRICT JSON (no prose):
{{"K": <integer 3-60>, "minimum_per_issue": <integer 1-10>, "reasoning": "<one sentence>"}}
"""

def render_top50(q, top_rows):
    lines = []
    for i, r in enumerate(top_rows.itertuples(), 1):
        comp = float(r.composite)
        role = str(r.citation_role) if hasattr(r, 'citation_role') else ""
        iss = str(r.matched_issue) if hasattr(r, 'matched_issue') else ""
        lines.append(f"  {i:>2}. {r.citation:<28}  score={comp:.2f}  issue={iss}  role={role[:20]}")
    return "\n".join(lines)

pass3_prompts, pass3_meta = [], []
for q in QIDS:
    top50 = (out_df[out_df.qid == q]
             .sort_values("composite", ascending=False)
             .head(50))
    issue_map_text = render_issue_map(qid_to_issues.get(q, []))
    candidates_block = render_top50(q, top50)
    p = PASS3_PROMPT.format(
        question=qid_to_query[q][:1500],
        issue_map=issue_map_text,
        candidates_block=candidates_block,
    )
    pass3_prompts.append(p)
    pass3_meta.append(q)
print(f"Pass-3 prompts: {len(pass3_prompts)}")
print(f"Sample (last 600 chars):\n{pass3_prompts[0][-600:]}")


## Phase 10 — Run Pass-3

In [ ]:
sp_pass3 = SamplingParams(temperature=0.6, top_p=0.95, top_k=20, min_p=0.0,
                          max_tokens=PASS3_MAX_TOKENS, seed=42)

print(f"Pass-3: {len(pass3_prompts)} prompts ...")
t0 = time.time()
outs3 = llm.chat([[{"role":"user","content":p}] for p in pass3_prompts], sampling_params=sp_pass3)
pass3_raws = [o.outputs[0].text for o in outs3]
print(f"Done in {(time.time()-t0):.1f}s")

del llm; gc.collect()
import torch; torch.cuda.empty_cache()

qid_to_K = {}
qid_to_min_per_issue = {}
for q, raw in zip(pass3_meta, pass3_raws):
    p = parse_json_block(raw) or {}
    try:
        K = int(p.get("K", 15))
    except Exception:
        K = 15
    try:
        m = int(p.get("minimum_per_issue", 1))
    except Exception:
        m = 1
    qid_to_K[q] = max(3, min(60, K))
    qid_to_min_per_issue[q] = max(1, min(10, m))

print(f"\nK predictions per query:")
for q in QIDS:
    print(f"  {q}: K={qid_to_K[q]}  min_per_issue={qid_to_min_per_issue[q]}  (gold={gold_totals[q]}, gold_in_topk={int(sb_per_query[q].is_gold.sum())})")


## Phase 11 — Pass-4 diversified selection + F1

In [ ]:
def select_picks(q_df, K, min_per_issue, score_floor):
    """Per-query selection: top-K by composite with per-issue floor; drop score < floor."""
    df = q_df.sort_values("composite", ascending=False).copy()
    df = df[df.composite >= score_floor]
    if len(df) == 0:
        return df.head(0)

    # Group by matched_issue (use "auto" / "none" / "" as their own buckets)
    by_issue = {}
    for issue, sub in df.groupby("matched_issue", sort=False):
        by_issue[issue] = sub.reset_index(drop=True)

    picks = []
    picked_dids = set()
    # First: take top min_per_issue from each issue bucket (excluding "none"/"" buckets)
    for issue, sub in by_issue.items():
        if issue in ("none", "") :
            continue
        for r in sub.head(min_per_issue).itertuples():
            if r.did not in picked_dids:
                picks.append(r)
                picked_dids.add(r.did)
    # Then: fill remaining slots by overall composite order
    remaining = K - len(picks)
    if remaining > 0:
        for r in df.itertuples():
            if r.did in picked_dids: continue
            picks.append(r)
            picked_dids.add(r.did)
            remaining -= 1
            if remaining <= 0: break
    # Trim back to K if over (shouldn't happen but safety)
    return pd.DataFrame([{c: getattr(r, c) for c in df.columns} for r in picks[:K]])

def f1(p, r): return 0.0 if (p+r)==0 else 2*p*r/(p+r)

v4_per_query = {"val_001":0.108,"val_002":0.047,"val_003":0.018,"val_004":0.076,"val_005":0.067,
                "val_006":0.109,"val_007":0.066,"val_008":0.013,"val_009":0.000,"val_010":0.065}
v53_per_query_k14 = {"val_001":0.167,"val_002":0.132,"val_003":0.064,"val_004":0.100,"val_005":0.091,
                     "val_006":0.056,"val_007":0.000,"val_008":0.100,"val_009":0.286,"val_010":0.080}

per_query = {}
all_picks_rows = []
print(f"{'qid':<8} {'gold':>5} {'in_topk':>8} {'K_pred':>7} {'picks':>6} {'correct':>8} "
      f"{'P':>5} {'R':>5} {'F1':>5}  {'v4':>5}  {'v5.3':>5}  delta_v4 delta_v5.3")

for q in QIDS:
    q_df = out_df[out_df.qid == q]
    K = qid_to_K[q]
    mp = qid_to_min_per_issue[q]
    picks_df = select_picks(q_df, K, mp, SCORE_FLOOR)
    pick_dids = set(picks_df.did) if "did" in picks_df.columns else set()
    correct = pick_dids & qid_to_gold[q]
    gold_total = gold_totals[q]
    gold_in_topk = int(sb_per_query[q].is_gold.sum())
    p = len(correct) / max(1, len(pick_dids))
    r = len(correct) / max(1, gold_total)
    f1_v = f1(p, r)
    per_query[q] = {"gold": gold_total, "gold_in_topk": gold_in_topk,
                    "K_predicted": K, "picks": len(pick_dids), "correct": len(correct),
                    "P": p, "R": r, "F1": f1_v}
    v4 = v4_per_query[q]
    v53 = v53_per_query_k14[q]
    d4 = f1_v - v4; d53 = f1_v - v53
    s4 = "+" if d4 > 0 else ""
    s53 = "+" if d53 > 0 else ""
    print(f"  {q:<6} {gold_total:>5} {gold_in_topk:>8} {K:>7} {len(pick_dids):>6} {len(correct):>8} "
          f"{p:>.3f} {r:>.3f} {f1_v:>.3f}  {v4:>.3f}  {v53:>.3f}  {s4}{d4:+.3f}  {s53}{d53:+.3f}")
    # tag picks rows for save
    for _, row in picks_df.iterrows():
        row_dict = row.to_dict()
        row_dict["correct"] = row_dict.get("did","") in qid_to_gold[q]
        all_picks_rows.append(row_dict)

macro_P = sum(per_query[q]["P"] for q in QIDS) / 10
macro_R = sum(per_query[q]["R"] for q in QIDS) / 10
macro_F = sum(per_query[q]["F1"] for q in QIDS) / 10
print(f"\nMACRO  P={macro_P:.3f}  R={macro_R:.3f}  F1={macro_F:.3f}")
print(f"\nBaselines:  v4 macro F1=0.057  |  v5.3 macro K-cap F1=0.107")
print(f"v6 macro F1 delta vs v4: {macro_F-0.057:+.3f}")
print(f"v6 macro F1 delta vs v5.3: {macro_F-0.107:+.3f}")


## Phase 12 — Save outputs

In [ ]:
out_df.to_parquet(OUT_DIR / "v6_scored.parquet", index=False)
if all_picks_rows:
    pd.DataFrame(all_picks_rows).to_parquet(OUT_DIR / "v6_picks.parquet", index=False)
with open(OUT_DIR / "v6_issue_maps.json", "w", encoding="utf-8") as f:
    json.dump(qid_to_issues, f, indent=2, ensure_ascii=False)
with open(OUT_DIR / "v6_metrics.json", "w") as f:
    json.dump({
        "config": {"TOP_K": TOP_K, "PASS1_SEEDS": PASS1_SEEDS,
                   "SCORE_FLOOR": SCORE_FLOOR},
        "macro_P": macro_P, "macro_R": macro_R, "macro_F1": macro_F,
        "per_query": per_query,
        "qid_to_K": qid_to_K,
        "qid_to_min_per_issue": qid_to_min_per_issue,
        "v4_macro_F1": 0.057, "v53_macro_K_cap_F1": 0.107,
    }, f, indent=2)
print(f"Saved 4 files to {OUT_DIR}")
print(f"\nv6 macro F1: {macro_F:.3f}  (v4: 0.057, v5.3: 0.107)")
